<a id="top"></a>
# Paper 2 — Master Ablation Notebook: Spatio-Chromatic Fourier Moments & Cross-Modal FiLM Grounding (`GeoFiLM`)

Notebook terpadu ini mengimplementasikan **4 Model Ablasi Mandiri untuk Paper 2** (**100% terlepas dari `CORN` dan `SA-Bridge`**), dengan pembagian data (*image-disjoint split*), *backbone* (`Florence-2-base-ft`, LoRA $r=32, lpha=64$), jadwal *optimizer*, dan sistem *auto-resume* yang **100% apple-to-apple** dengan eksperimen sebelumnya.

* **Folder Eksperimen Output:** `/content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Paper2_GeoFiLM_r32`
* **Sakelar Tunggal (`VARIANT_KIND` di Cell `[A12]`):**
  1. `"M1_GeoPE"` — **+ 2D Learnable Fourier Coordinate Encoding & Continuous Spatial Moments $(oldsymbol{\mu}_{	ext{pos}}, oldsymbol{\Sigma}_{	ext{pos}})$** *(Menargetkan kebutaan koordinat lokasi `abnormality_location`, `landmark_location`, `instrument_location`)*.
  2. `"M2_GeoDiff"` — **+ Orthogonal Foreground-Background Chromatic/Tissue Differential Stream $(\mathbf{h}_{	ext{fg}} - \mathbf{h}_{	ext{bg}})$** *(Menargetkan `polyp_removal_status`, `abnormality_color`, & `abnormality_presence`)*.
  3. `"M3_StaticGate"` — **+ Unconditional Scalar Gate Modulation $(\gamma_{	ext{glob}}, \gamma_{	ext{spat}} \in \mathbb{R}^1)$** *(Ablasi kontrol gerbang statis tanpa pengondisian pertanyaan)*.
  4. `"M4_GeoFiLM"` — **Full Proposed Model: Query-Conditioned Cross-Modal Channel-Wise FiLM $(oldsymbol{\gamma}, oldsymbol{eta} \in \mathbb{R}^{768})$** *(Menghilangkan trade-off antar-kategori & menyelaraskan penalaran komposisional Level 1, Level 2, dan Level 3 untuk mengungguli `Gautam Qwen2.5-VL-7B`)*.

---

### 🧭 Navigasi Cepat Antar Bagian:
| Bagian | Deskripsi Singkat | Jalur Eksekusi |
| :--- | :--- | :--- |
| **[PART A — Persiapan & Arsitektur GeoFiLM](#part-a)** | Environment, Data Split, Arsitektur 4 Varian `GeoFiLMModule` (`[A1]`–`[A14]`) | **Wajib** (Jalankan selalu) |
| **[SMOKE TEST Pra-Training](#smoke-test)** | Verifikasi forward, aliran gradien ke DaViT, Cross-Modal FiLM, & fp16 (`[A15]`) | *Direkomendasikan (~5 detik)* |
| **[PART B — FRESH RUN](#part-b)** | Reseed bersih & alokasi folder run sesuai `VARIANT_KIND` (`[B1]`–`[B2]`) | **Wajib jika Run Baru** |
| **[PART C — RESUME](#part-c)** | Pulihkan bobot, modul GeoFiLM, & optimizer jika Colab terputus (`[C1]`) | ⚠️ **HANYA jika terputus** |
| **[PART D — TRAINING LOOP (5 Epoch)](#part-d)** | Training utama $\mathcal{L}_{	ext{VQA}} + \lambda_{	ext{reg}}\mathcal{L}_{	ext{struct}}$, validasi per 500 step, auto-save (`[D1]`–`[D4]`) | **Wajib** (Jalankan training) |
| **[PART E — INFERENSI 15.955 TEST SET & EVALUASI](#part-e)** | Inferensi `beam=3` pada 15.955 QA uji & tabel komparasi `A0` vs `M1`–`M4` vs `Gautam 7B` (`[E1]`–`[E4]`) | Jalankan setelah training selesai |

# PART A — COMMON SETUP & ARSITEKTUR GEOFILM (Jalankan untuk FRESH, RESUME, maupun EVAL)
<a id="part-a"></a>
> [⬆️ Kembali ke Navigasi Cepat](#top) | [Lompat ke SMOKE TEST](#smoke-test) | [Lompat ke PART B (Fresh Run)](#part-b) | [Lompat ke PART D (Training)](#part-d) | [Lompat ke PART E (Evaluasi Test Set)](#part-e)

# [A1] Environment — install paket

In [1]:
# [A1] Environment — install paket & restart runtime sekali
!pip install -q transformers==4.49.0 peft "torchao>=0.16.0" rouge-score sacrebleu nltk scikit-learn
import transformers, peft
print("transformers:", transformers.__version__, "| peft:", peft.__version__)
print("\n>>> JIKA BARU INSTAL PAKET, RESTART RUNTIME SEKARANG, lalu lanjutkan mulai dari Cell [A2] <<<")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers: 4.49.0 | peft: 0.21.2

>>> JIKA BARU INSTAL PAKET, RESTART RUNTIME SEKARANG, lalu lanjutkan mulai dari Cell [A2] <<<


# [A2] Import & cek versi pustaka

In [1]:
# [A2] Import & cek versi pustaka
import transformers, peft, torch
print("transformers:", transformers.__version__, "| peft:", peft.__version__, "| torch:", torch.__version__)

transformers: 4.49.0 | peft: 0.21.2 | torch: 2.11.0+cu130


# [A3] Mount Google Drive

In [2]:
# [A3] Mount Google Drive (otomatis dilewati jika berjalan di mesin lokal Windows/Linux)
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
print("Running on Colab:", IS_COLAB)

Mounted at /content/drive
Running on Colab: True


# [A4] Konfigurasi path,

In [3]:
# [A4] Konfigurasi path, seed, & caching gambar ke disk lokal (/content/all_images di Colab)
from pathlib import Path
import json, re, math, random, os, shutil, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

if IS_COLAB:
    BASE = Path("/content/drive/MyDrive/DOKTORAL/KvasirVQAX1")
    LOCAL = Path("/content/all_images")
else:
    BASE = Path(r"g:/My Drive/DOKTORAL/KvasirVQAX1")
    LOCAL = BASE / "all_images"

SRC_DIR  = BASE / "all_images"
ZIP_PATH = BASE / "all_images.zip"

t0_img = time.time()
if IS_COLAB and (not LOCAL.exists() or len(os.listdir(LOCAL)) < 6449):
    LOCAL.mkdir(parents=True, exist_ok=True)
    if ZIP_PATH.exists():
        print(f"Menyalin arsip {ZIP_PATH.name} dari Google Drive ke disk lokal VM...")
        shutil.copy2(ZIP_PATH, "/content/all_images.zip")
        print("Mengekstrak gambar ke disk lokal (/content/all_images)...")
        os.system("unzip -q -o /content/all_images.zip -d /content/all_images")
        if os.path.exists("/content/all_images.zip"):
            os.remove("/content/all_images.zip")
        print(f"✓ Selesai ekstrak dalam {time.time() - t0_img:.1f} detik! Total gambar: {len(os.listdir(LOCAL))}")
    else:
        from tqdm.auto import tqdm
        src_files = [f for f in os.listdir(SRC_DIR) if not f.startswith(".")]
        loc_files = set(os.listdir(LOCAL))
        missing_files = [f for f in src_files if f not in loc_files]
        for f in tqdm(missing_files, desc="Salin Gambar", unit="file"):
            shutil.copy2(SRC_DIR / f, LOCAL / f)
        print(f"✓ Selesai salin dalam {time.time() - t0_img:.1f} detik! Total gambar: {len(os.listdir(LOCAL))}")
else:
    print(f"✓ Folder gambar siap: {LOCAL} ({len(os.listdir(LOCAL)) if LOCAL.exists() else 0} file).")

IMG_DIR    = LOCAL
A0_R32_DIR = BASE / "experiments/Stage2_CORAL_linear/prediction_rank32/20260916-060029_VQAonly-r32_a0.0_s42"
AUDIT      = BASE / "audit_stage0/stage0_audit.json"
SPLIT_PATH = BASE / "experiments/Stage2_CORAL_linear/val_split_images.json"
BASE_ID    = "microsoft/Florence-2-base-ft"
OUTDIR     = BASE / "experiments/Paper2_GeoFiLM_r32"
OUTDIR.mkdir(parents=True, exist_ok=True)

if SPLIT_PATH.exists() and not (OUTDIR / "val_split_images.json").exists():
    shutil.copy2(SPLIT_PATH, OUTDIR / "val_split_images.json")

for name, p in [("all_images", IMG_DIR), ("A0_vqaonly_r32", A0_R32_DIR), ("split", SPLIT_PATH)]:
    print("OK  " if p.exists() else "MISS", f"{p}")
print("OK   OUTDIR ->", OUTDIR)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "")

Menyalin arsip all_images.zip dari Google Drive ke disk lokal VM...
Mengekstrak gambar ke disk lokal (/content/all_images)...
✓ Selesai ekstrak dalam 11.4 detik! Total gambar: 6449
OK   /content/all_images
OK   /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Stage2_CORAL_linear/prediction_rank32/20260916-060029_VQAonly-r32_a0.0_s42
OK   /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Stage2_CORAL_linear/val_split_images.json
OK   OUTDIR -> /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Paper2_GeoFiLM_r32
device: cuda NVIDIA A100-SXM4-40GB


# [A5] Warisi Hyperparameter & konfigurasi LoRA r=32

In [4]:
# [A5] Warisi Hyperparameter & konfigurasi LoRA r=32 langsung dari Baseline A0 (VQAonly-r32)
A0_CFG_PATH  = A0_R32_DIR / "config.json"
A0_LORA_PATH = A0_R32_DIR / "checkpoints/final/adapter_config.json"

if A0_CFG_PATH.exists():
    a0_meta = json.loads(A0_CFG_PATH.read_text())
    HP = a0_meta["hp"]
    print(f"Loaded HP from A0 VQA-only r=32 ({A0_R32_DIR.name}):")
else:
    HP = {
        "lr": 1e-05, "batch_size": 16, "grad_accum": 2, "epochs": 5.0,
        "warmup_ratio": 0.1, "weight_decay": 0.01, "scheduler": "linear",
        "fp16": True, "bf16": False
    }
    print("Loaded canonical A0 VQA-only r=32 HP:")

for k, v in HP.items():
    print(f"  {k:14s} {v}")

if A0_LORA_PATH.exists():
    lora_cfg = json.loads(A0_LORA_PATH.read_text())
else:
    lora_cfg = {
        "r": 32, "lora_alpha": 64, "lora_dropout": 0.05,
        "target_modules": ["q_proj", "qkv", "proj", "v_proj"],
        "bias": "none", "task_type": "SEQ_2_SEQ_LM"
    }

print("\nLoRA r=32 config (from A0 VQA-only r=32):")
for k in ["r", "lora_alpha", "lora_dropout", "target_modules", "bias", "task_type"]:
    print(f"  {k:16s} {lora_cfg.get(k)}")
assert int(lora_cfg["r"]) == 32 and int(lora_cfg["lora_alpha"]) == 64, f"Expected r=32, got {lora_cfg}"

Loaded HP from A0 VQA-only r=32 (20260916-060029_VQAonly-r32_a0.0_s42):
  lr             1e-05
  batch_size     16
  grad_accum     2
  epochs         5.0
  warmup_ratio   0.1
  weight_decay   0.01
  scheduler      linear
  fp16           True
  bf16           False

LoRA r=32 config (from A0 VQA-only r=32):
  r                32
  lora_alpha       64
  lora_dropout     0.05
  target_modules   ['qkv', 'q_proj', 'proj', 'v_proj']
  bias             none
  task_type        SEQ_2_SEQ_LM


# [A6] Muat dataset train & terapkan Image-Disjoint Validation Split

In [5]:
# [A6] Muat dataset train & terapkan Image-Disjoint Validation Split yang 100% identik
from datasets import load_from_disk

ds_tr = load_from_disk(str(BASE / "train"))
tr = ds_tr.to_pandas()
print("Total train rows:", len(tr))

strip_jpg = lambda s: re.sub(r"\.(jpg|jpeg|png)$", "", str(s), flags=re.I)
tr["_img"] = tr["img_id"].map(strip_jpg)

assert SPLIT_PATH.exists(), f"Split file tidak ditemukan di {SPLIT_PATH}"
val_imgs = set(json.loads(SPLIT_PATH.read_text())["val_images"])
tr["_split"] = np.where(tr["_img"].isin(val_imgs), "val", "train")

overlap = set(tr.loc[tr._split == "train", "_img"]) & set(tr.loc[tr._split == "val", "_img"])
assert len(overlap) == 0, "Image overlap antara train dan val harus 0!"

for s in ["train", "val"]:
    g = tr[tr._split == s]
    print(f"  {s:5s} rows={len(g):7d}  unique_images={g._img.nunique():5d}")

Total train rows: 143594
  train rows= 129023  unique_images= 5590
  val   rows=  14571  unique_images=  622


# [A7] Dataset VQAGeneralDS & fungsi collate

In [6]:
# [A7] Dataset VQAGeneralDS & fungsi collate (Tanpa label ordinal CORN — Murni End-to-End VQA!)
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from unittest.mock import patch
from transformers import AutoProcessor, AutoModelForCausalLM
from transformers.dynamic_module_utils import get_imports

PROMPT_PREFIX = "<VQA> "

def _skip_flash_proc(f):
    return [i for i in get_imports(f) if i != "flash_attn"]

with patch("transformers.dynamic_module_utils.get_imports", _skip_flash_proc):
    processor = AutoProcessor.from_pretrained(BASE_ID, trust_remote_code=True)

class VQAGeneralDS(Dataset):
    def __init__(self, df):
        self.q = df["question"].astype(str).str.strip().tolist()
        self.a = df["answer"].astype(str).str.strip().tolist()
        self.p = [IMG_DIR / f"{i}.jpg" for i in df["_img"]]
    def __len__(self):
        return len(self.q)
    def __getitem__(self, i):
        p = self.p[i]
        try:
            img = Image.open(p).convert("RGB")
        except Exception:
            drive_src = BASE / "all_images" / p.name
            if drive_src.exists():
                shutil.copy2(drive_src, p)
                img = Image.open(p).convert("RGB")
            else:
                raise
        return {"q": PROMPT_PREFIX + self.q[i], "a": self.a[i], "img": img}

def collate(batch):
    enc = processor(text=[b["q"] for b in batch],
                    images=[b["img"] for b in batch],
                    return_tensors="pt", padding=True)
    # Tokenisasi dengan BOS/EOS aktif untuk mencegah bug off-by-one shift_tokens_right
    lab = processor.tokenizer([b["a"] for b in batch], return_tensors="pt",
                              padding=True, return_token_type_ids=False).input_ids
    lab[lab == processor.tokenizer.pad_token_id] = -100
    return {
        "input_ids": enc["input_ids"],
        "pixel_values": enc["pixel_values"],
        "attention_mask": enc.get("attention_mask"),
        "labels": lab,
    }

ds_train = VQAGeneralDS(tr[tr._split == "train"])
ds_val   = VQAGeneralDS(tr[tr._split == "val"])
print(f"Dataset siap | Train: {len(ds_train)} sampel | Val: {len(ds_val)} sampel")

b_test = collate([ds_train[i] for i in range(4)])
bos_id = processor.tokenizer.bos_token_id
assert (b_test["labels"][:, 0] == bos_id).all(), "Token ke-0 pada labels harus bos_token_id!"
print(f"[VERIFIED] Label token ke-0 = BOS ({bos_id}) & bebas bug prefix-shift.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

processing_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base-ft:
- processing_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/root/.cache/huggingface/modules/transformers_modules/microsoft/Florence-2-base-ft/f6c1a25888ffc1d945ee8a1a77ac833c7303d46e/processing_florence2.py:515: SyntaxWarning: invalid escape sequence '\d'
  PATTERN: 'r<time_(\d+)><time_(\d+)>([a-zA-Z0-9 ]+)'


tokenizer_config.json:   0%|          | 0.00/34.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

configuration_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base-ft:
- configuration_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Dataset siap | Train: 129023 sampel | Val: 14571 sampel
[VERIFIED] Label token ke-0 = BOS (0) & bebas bug prefix-shift.


## Arsitektur `GeoFiLMModule` (Mencakup 4 Varian Ablasi: `M1_GeoPE`, `M2_GeoDiff`, `M3_StaticGate`, `M4_GeoFiLM`)

Seluruh komponen di bawah ini **100% *end-to-end learnable***, tidak menggunakan *hard-coded routing*, dan **tidak menggunakan `CORN` maupun `SA-Bridge`**:

1. **`M1_GeoPE` (`use_geope=True, use_diff=False, gate_mode="additive"`):**
   - Mengekstrak peta *self-supervised visual saliency* $	ilde{\mathbf{a}} \in \Delta^{575}$ pada grid $24 	imes 24$.
   - Menyuntikkan **Learnable 2D Fourier Positional Encoding** $oldsymbol{\Phi}_{	ext{2D}}(\mathbf{P}) \in \mathbb{R}^{576 	imes D}$ langsung ke setiap *spatial patch token* (mengembalikan kesadaran koordinat 2D pasca-`image_projection` seperti M-RoPE pada Qwen2.5-VL).
   - Menghitung **9D Continuous 2D Spatio-Geometric Moments** ($oldsymbol{\mu}_{	ext{pos}} = [ar{x}, ar{y}]$, kovarians $oldsymbol{\Sigma}_{	ext{pos}} = [\sigma_{xx}, \sigma_{yy}, \sigma_{xy}]$, dan 4 statistik konsentrasi saliency) $	o$ dipetakan via *Learnable Fourier MLP* menjadi vektor geometri $\mathbf{e}_{	ext{geom}} \in \mathbb{R}^D$.
2. **`M2_GeoDiff` (`use_geope=True, use_diff=True, gate_mode="additive"`):**
   - Menambahkan **Chromatic-Tissue Differential Contrast Stream** $\mathbf{e}_{	ext{diff}} \in \mathbb{R}^D$ yang memproyeksikan selisih fitur jaringan *foreground* terhadap mukosa *background* $(\mathbf{h}_{	ext{fg}} - \mathbf{h}_{	ext{bg}}) \oplus (\mathbf{h}_{	ext{fg}} - \mathbf{t}_0)$ melalui *low-rank bottleneck* ($2D 	o 64 	o D$).
   - Dilengkapi **Orthogonal Subspace Regularizer** $\mathcal{L}_{	ext{orth}} = \|\mathbf{W}_{	ext{geom}}^	op \mathbf{W}_{	ext{diff}}\|_F^2$ agar informasi lokasi ($\mathbf{e}_{	ext{geom}}$) dan informasi kontras mukosa/status reseksi ($\mathbf{e}_{	ext{diff}}$) menempati sub-ruang yang saling tegak lurus (*zero interference*).
3. **`M3_StaticGate` (`use_geope=True, use_diff=True, gate_mode="scalar"`):**
   - Memodulasi token visual menggunakan gerbang skalar $	anh(\gamma_{	ext{glob}}), 	anh(\gamma_{	ext{spat}}) \in \mathbb{R}^1$ tanpa pengondisian konteks kueri $\mathbf{Q}$.
4. **`M4_GeoFiLM` (`use_geope=True, use_diff=True, gate_mode="film"` — Full Proposed Model):**
   - Menggunakan **Learnable Cross-Modal FiLM Controller**: representasi pertanyaan $\mathbf{Q}$ melakukan *Cross-Attention* ke token ringkasan $[\mathbf{t}_0 ; \mathbf{e}_{	ext{geom}} ; \mathbf{e}_{	ext{diff}}]$ untuk memprediksi vektor gerbang per-kanal $oldsymbol{\gamma}_{	ext{glob}}, oldsymbol{eta}_{	ext{glob}}, oldsymbol{\gamma}_{	ext{spat}}, oldsymbol{eta}_{	ext{spat}} \in (-1, 1)^{768}$.

# [A8] Definisi PyTorch GeoFiLMModule

In [7]:
# [A8] Definisi PyTorch GeoFiLMModule (Mendukung M1_GeoPE, M2_GeoDiff, M3_StaticGate, M4_GeoFiLM)
class GeoFiLMModule(nn.Module):
    def __init__(self, d_in=768, d_fourier=32, d_rank=64,
                 use_geope=True, use_diff=True, gate_mode="film"):
        super().__init__()
        assert gate_mode in ("additive", "scalar", "film"), f"Unknown gate_mode: {gate_mode}"
        self.d_in = d_in
        self.d_fourier = d_fourier
        self.d_rank = d_rank
        self.use_geope = use_geope
        self.use_diff = use_diff
        self.gate_mode = gate_mode
        self.scale = 1.0 / math.sqrt(d_in)

        # 1. Self-Supervised Multi-Scale Saliency Scorer
        self.norm_in = nn.LayerNorm(d_in)
        self.sal_proj = nn.Sequential(
            nn.Linear(d_in, 128),
            nn.GELU(),
            nn.Linear(128, 1)
        )

        # Precompute normalized 2D grid coordinates [-1, 1]^2 for 24x24 = 576 patches
        gy, gx = torch.meshgrid(
            torch.linspace(-1.0, 1.0, 24),
            torch.linspace(-1.0, 1.0, 24),
            indexing="ij"
        )
        coords_2d = torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=-1)  # (576, 2)
        self.register_buffer("coords_2d", coords_2d, persistent=False)

        # 2. Modul M1: Dense 2D Learnable Fourier Positional Encoding + 9D Spatio-Geometric Moment MLP
        self.fourier_B_patch = nn.Parameter(torch.randn(2, d_fourier) * 0.5)
        self.patch_pe_mlp = nn.Sequential(
            nn.Linear(2 * d_fourier, d_in),
            nn.LayerNorm(d_in)
        )
        self.pe_scale = nn.Parameter(torch.tensor(0.0))  # tanh(0) = 0 warm start

        # 9D moments: [mu_x, mu_y, cov_xx, cov_yy, cov_xy, sal_mean, sal_std, sal_max, sal_entropy]
        self.fourier_B_mom = nn.Parameter(torch.randn(9, d_fourier) * 0.3)
        self.geom_mlp = nn.Sequential(
            nn.Linear(9 + 2 * d_fourier, 128),
            nn.GELU(),
            nn.Linear(128, d_in, bias=False),
            nn.LayerNorm(d_in)
        )
        self.W_geom_proj = self.geom_mlp[2]  # Linear(128, d_in) untuk perhitungan L_orth

        # 3. Modul M2: Chromatic-Tissue Foreground-Background Differential Contrast Stream
        if self.use_diff:
            self.diff_down = nn.Sequential(
                nn.LayerNorm(2 * d_in),
                nn.Linear(2 * d_in, d_rank),
                nn.GELU()
            )
            self.diff_up = nn.Linear(d_rank, d_in, bias=False)
            self.diff_norm = nn.LayerNorm(d_in)

        # 4. Modul M3 / M4: Gating Mechanism
        if self.gate_mode == "additive":
            # M1_GeoPE & M2_GeoDiff: Direct residual bridge dengan satu scalar warm-start
            self.bridge_alpha = nn.Parameter(torch.zeros(1))
        elif self.gate_mode == "scalar":
            # M3_StaticGate: Input-agnostic (Query-blind) decoupled global/spatial & channel-wise static gates
            self.gate_global  = nn.Parameter(torch.zeros(1, 1, d_in))
            self.gate_spatial = nn.Parameter(torch.zeros(1, 1, d_in))
            self.scale_global = nn.Parameter(torch.zeros(1, 1, d_in))
            self.scale_spatial= nn.Parameter(torch.zeros(1, 1, d_in))
        elif self.gate_mode == "film":
            # Untuk M4_GeoFiLM: Cross-Modal Attention antara Query (Q) dan Visual Context [t0, e_geom, e_diff]
            self.q_norm = nn.LayerNorm(d_in)
            self.kv_norm = nn.LayerNorm(d_in)
            self.cross_q = nn.Linear(d_in, 128)
            self.cross_k = nn.Linear(d_in, 128)
            self.cross_v = nn.Linear(d_in, 128)
            self.film_head = nn.Sequential(
                nn.Linear(128 + d_in, 256),
                nn.GELU(),
                nn.Linear(256, 4 * d_in)  # [gamma_glob, beta_glob, gamma_spat, beta_spat]
            )
            # Zero-initialize layer terakhir agar pada step 0 identik dengan unmodulated backbone
            nn.init.zeros_(self.film_head[-1].weight)
            nn.init.zeros_(self.film_head[-1].bias)

    def _compute_saliency_and_moments(self, feats):
        B, N, D = feats.shape
        cls_tok = feats[:, 0, :]
        patches = feats[:, 1:, :]
        M = patches.shape[1]

        p_norm = self.norm_in(patches)
        s = self.sal_proj(p_norm).squeeze(-1) * self.scale          # (B, 576)
        a = torch.softmax(s.float(), dim=1).to(feats.dtype)         # (B, 576)
        a_unsq = a.unsqueeze(-1)                                    # (B, 576, 1)

        # Foreground & Background representations
        fg_mean = (a_unsq * patches).sum(dim=1)                     # (B, D)
        bg_w = (1.0 - torch.sigmoid(s.float() / self.scale))
        bg_w = (bg_w / bg_w.sum(dim=1, keepdim=True).clamp(min=1e-6)).to(feats.dtype).unsqueeze(-1)
        bg_mean = (bg_w * patches).sum(dim=1)                       # (B, D)

        # 2D Spatial Coordinates (handle dynamic token count if needed)
        coords = self.coords_2d[:M, :].to(feats.dtype)              # (M, 2)
        af = a.float()                                              # (B, M)
        cf = coords.float()                                         # (M, 2)

        # 1st-Order Spatial Moment (Centroid: mu_x, mu_y)
        mu_pos = torch.matmul(af, cf)                               # (B, 2)

        # 2nd-Order Spatial Moment (Covariance: cov_xx, cov_yy, cov_xy)
        dx = cf[:, 0].unsqueeze(0) - mu_pos[:, 0:1]                 # (B, M)
        dy = cf[:, 1].unsqueeze(0) - mu_pos[:, 1:2]                 # (B, M)
        cov_xx = (af * (dx ** 2)).sum(dim=1, keepdim=True)
        cov_yy = (af * (dy ** 2)).sum(dim=1, keepdim=True)
        cov_xy = (af * (dx * dy)).sum(dim=1, keepdim=True)

        # 0th-Order Saliency Intensity Statistics
        gate_p = torch.sigmoid(s.float() / self.scale)
        sal_mean = gate_p.mean(dim=1, keepdim=True)
        sal_std  = gate_p.std(dim=1, keepdim=True)
        sal_max  = torch.tanh(s.float().max(dim=1, keepdim=True)[0])
        af_clamped = af.clamp(min=1e-8)
        sal_ent  = -(af_clamped * torch.log(af_clamped)).sum(dim=1, keepdim=True) / max(math.log(M), 1.0)

        moments_9d = torch.cat([
            mu_pos, cov_xx, cov_yy, cov_xy,
            sal_mean, sal_std, sal_max, sal_ent
        ], dim=1).to(feats.dtype)                                   # (B, 9)

        return cls_tok, patches, a_unsq, fg_mean, bg_mean, moments_9d, coords, sal_ent.mean()

    def orth_And_sal_loss(self, sal_ent_mean):
        # 1. Orthogonality loss antara ruang geometri (W_geom) dan ruang kontras jaringan (W_diff)
        loss_orth = torch.tensor(0.0, device=self.fourier_B_patch.device)
        if self.use_diff:
            # W_geom_proj.weight: (D, 128), diff_up.weight: (D, d_rank)
            w_g = F.normalize(self.W_geom_proj.weight.float(), dim=0)
            w_d = F.normalize(self.diff_up.weight.float(), dim=0)
            gram = torch.matmul(w_g.t(), w_d)                       # (128, d_rank)
            loss_orth = (gram ** 2).mean()
        # 2. Mild entropy regularization agar peta saliency fokus (tidak kolaps jadi uniform=1.0 atau 1-pixel=0.0)
        # Target entropi moderat ~ 0.55
        loss_ent = ((sal_ent_mean.float() - 0.55) ** 2)
        return loss_orth + 0.1 * loss_ent

    def forward_bridge(self, feats, query_embeds=None, query_mask=None):
        if feats.dim() == 2 or feats.shape[1] <= 1:
            return feats, torch.tensor(0.0, device=feats.device)

        cls_tok, patches, a_unsq, fg_mean, bg_mean, moments_9d, coords, sal_ent_mean = self._compute_saliency_and_moments(feats)
        B, M, D = patches.shape

        # A. Dense 2D Fourier Positional Encoding pada seluruh 576 patch
        proj_2d = 2.0 * math.pi * torch.matmul(coords.float(), self.fourier_B_patch.float())
        fourier_2d = torch.cat([torch.sin(proj_2d), torch.cos(proj_2d)], dim=-1).to(feats.dtype)
        pe_2d = self.patch_pe_mlp(fourier_2d).unsqueeze(0)          # (1, M, D)
        pe_w = torch.tanh(self.pe_scale.to(feats.dtype))

        # B. 9D Spatio-Geometric Moment Embedding (e_geom)
        proj_mom = 2.0 * math.pi * torch.matmul(moments_9d.float(), self.fourier_B_mom.float())
        fourier_mom = torch.cat([moments_9d.float(), torch.sin(proj_mom), torch.cos(proj_mom)], dim=-1).to(feats.dtype)
        e_geom = self.geom_mlp(fourier_mom)                         # (B, D)

        # C. Chromatic-Tissue Differential Contrast Embedding (e_diff)
        if self.use_diff:
            diff_in = torch.cat([fg_mean - bg_mean, fg_mean - cls_tok], dim=-1)
            e_diff = self.diff_norm(self.diff_up(self.diff_down(diff_in)))  # (B, D)
        else:
            e_diff = torch.zeros_like(e_geom)

        e_total = e_geom + e_diff                                   # (B, D)

        # D. Token Modulation sesuai gate_mode (additive / scalar / film)
        if self.gate_mode == "additive":
            # M1_GeoPE & M2_GeoDiff: Uniform additive injection (tanpa pemisahan gerbang per-kanal)
            alpha_w = torch.tanh(self.bridge_alpha.to(feats.dtype))
            tok0_new = cls_tok.unsqueeze(1) + alpha_w * e_total.unsqueeze(1)
            patches_new = patches + pe_w * pe_2d + alpha_w * (M * a_unsq) * e_total.unsqueeze(1)
        elif self.gate_mode == "scalar":
            # M3_StaticGate: Channel-wise static affine gate (Query-blind, sama untuk semua pertanyaan)
            gam_g = torch.tanh(self.scale_global.to(feats.dtype))
            bet_g = torch.tanh(self.gate_global.to(feats.dtype))
            gam_s = torch.tanh(self.scale_spatial.to(feats.dtype))
            bet_s = torch.tanh(self.gate_spatial.to(feats.dtype))
            tok0_new = (1.0 + gam_g) * cls_tok.unsqueeze(1) + bet_g * e_total.unsqueeze(1)
            spat_inject = (M * a_unsq) * e_geom.unsqueeze(1) + a_unsq * e_diff.unsqueeze(1)
            patches_new = (1.0 + gam_s) * patches + pe_w * pe_2d + bet_s * spat_inject
        else:
            # Cross-Modal FiLM Modulation (M4_GeoFiLM)
            if query_embeds is not None and query_embeds.shape[0] == B:
                q_emb = query_embeds.to(feats.dtype)
                if query_mask is not None and query_mask.shape[:2] == q_emb.shape[:2]:
                    qm = query_mask.to(feats.dtype).unsqueeze(-1)
                    q_pool = (q_emb * qm).sum(dim=1) / qm.sum(dim=1).clamp(min=1.0)
                else:
                    q_pool = q_emb.mean(dim=1)
            else:
                q_pool = cls_tok

            # Cross-attention antara q_pool (Query) dan 3 visual context tokens [cls_tok, e_geom, e_diff]
            kv_tokens = self.kv_norm(torch.stack([cls_tok, e_geom, e_diff], dim=1))  # (B, 3, D)
            q_vec = self.cross_q(self.q_norm(q_pool)).unsqueeze(1)                   # (B, 1, 128)
            k_mat = self.cross_k(kv_tokens)                                          # (B, 3, 128)
            v_mat = self.cross_v(kv_tokens)                                          # (B, 3, 128)

            attn_scores = torch.matmul(q_vec.float(), k_mat.float().transpose(-1, -2)) / math.sqrt(128.0)
            attn_probs = torch.softmax(attn_scores, dim=-1).to(feats.dtype)          # (B, 1, 3)
            c_joint = torch.matmul(attn_probs, v_mat).squeeze(1)                     # (B, 128)

            film_params = self.film_head(torch.cat([c_joint, e_total], dim=-1))      # (B, 4*D)
            gam_g, bet_g, gam_s, bet_s = torch.chunk(film_params, 4, dim=-1)
            gam_g = torch.tanh(gam_g).unsqueeze(1)                                   # (B, 1, D)
            bet_g = torch.tanh(bet_g).unsqueeze(1)                                   # (B, 1, D)
            gam_s = torch.tanh(gam_s).unsqueeze(1)                                   # (B, 1, D)
            bet_s = torch.tanh(bet_s).unsqueeze(1)                                   # (B, 1, D)

            tok0_new = (1.0 + gam_g) * cls_tok.unsqueeze(1) + bet_g * e_total.unsqueeze(1)
            spat_inject = (M * a_unsq) * e_geom.unsqueeze(1) + a_unsq * e_diff.unsqueeze(1)
            patches_new = (1.0 + gam_s) * patches + pe_w * pe_2d + bet_s * spat_inject

        bridged = torch.cat([tok0_new, patches_new], dim=1)
        reg_loss = self.orth_And_sal_loss(sal_ent_mean)
        return bridged, reg_loss

# Uji cepat keempat konfigurasi ablasi
_x_dummy = torch.randn(4, 577, 768)
_q_dummy = torch.randn(4, 14, 768)
for _vname, _cfg in [
    ("M1_GeoPE",      dict(use_geope=True, use_diff=False, gate_mode="additive")),
    ("M2_GeoDiff",    dict(use_geope=True, use_diff=True,  gate_mode="additive")),
    ("M3_StaticGate", dict(use_geope=True, use_diff=True,  gate_mode="scalar")),
    ("M4_GeoFiLM",    dict(use_geope=True, use_diff=True,  gate_mode="film")),
]:
    _m = GeoFiLMModule(768, **_cfg)
    _out, _rl = _m.forward_bridge(_x_dummy, _q_dummy)
    print(f"{_vname:15s} | params: {sum(p.numel() for p in _m.parameters()):7d} | out: {tuple(_out.shape)} | identity_at_init: {torch.allclose(_out, _x_dummy, atol=1e-6)}")

M1_GeoPE        | params:  261219 | out: (4, 577, 768) | identity_at_init: True
M2_GeoDiff      | params:  413347 | out: (4, 577, 768) | identity_at_init: True
M3_StaticGate   | params:  416418 | out: (4, 577, 768) | identity_at_init: True
M4_GeoFiLM      | params: 1730850 | out: (4, 577, 768) | identity_at_init: True


# [A9] SAKELAR TUNGGAL VARIANT_KIND

In [8]:
# [A9] SAKELAR TUNGGAL VARIANT_KIND — Pilih Varian Ablasi Paper 2 yang Akan Dijalankan
# Pilihan:
#   "M1_GeoPE"      -> Ablasi 1: 2D Fourier Positional Encoding + 9D Spatio-Geometric Moments
#   "M2_GeoDiff"    -> Ablasi 2: M1 + Orthogonal Chromatic-Tissue Differential Stream
#   "M3_StaticGate" -> Ablasi 3: M2 + Input-Agnostic Scalar Gate
#   "M4_GeoFiLM"    -> Full Proposed Model: M2 + Query-Conditioned Cross-Modal Channel-Wise FiLM
VARIANT_KIND = "M3_StaticGate"

ABLATION_CONFIGS = {
    "M1_GeoPE":      dict(use_geope=True, use_diff=False, gate_mode="additive"),
    "M2_GeoDiff":    dict(use_geope=True, use_diff=True,  gate_mode="additive"),
    "M3_StaticGate": dict(use_geope=True, use_diff=True,  gate_mode="scalar"),
    "M4_GeoFiLM":    dict(use_geope=True, use_diff=True,  gate_mode="film"),
}
assert VARIANT_KIND in ABLATION_CONFIGS, f"VARIANT_KIND tidak dikenal: {VARIANT_KIND}"
UNFREEZE_IMG_PROJ = True

def make_head(d_vis=768):
    return GeoFiLMModule(d_in=d_vis, **ABLATION_CONFIGS[VARIANT_KIND]).to(dev)

print("Active VARIANT_KIND :", VARIANT_KIND)
print("Module Config       :", ABLATION_CONFIGS[VARIANT_KIND])

Active VARIANT_KIND : M3_StaticGate
Module Config       : {'use_geope': True, 'use_diff': True, 'gate_mode': 'scalar'}


# [A10] Build model Florence-2 + LoRA r=32

In [9]:
# [A10] Build model Florence-2 + LoRA r=32 + pasang hook get_input_embeddings & _encode_image
from peft import LoraConfig, get_peft_model
import transformers.dynamic_module_utils as dmu
orig_get_imports = dmu.get_imports

def _skip_flash(name, *args, **kwargs):
    if "flash_attn" in str(name):
        return None
    return orig_get_imports(name, *args, **kwargs)

LORA_R     = 32
LORA_ALPHA = LORA_R * 2

def _unwrap(m):
    seen = set()
    def walk(x, depth=0):
        if depth > 6 or id(x) in seen: return None
        seen.add(id(x))
        if "_encode_image" in type(x).__dict__: return x
        for name in ("base_model", "model"):
            child = getattr(x, name, None)
            if child is not None and child is not x:
                r = walk(child, depth + 1)
                if r is not None: return r
        return None
    found = walk(m)
    assert found is not None, "could not locate the module defining _encode_image"
    return found

def build_model():
    with patch("transformers.dynamic_module_utils.get_imports", _skip_flash):
        m = AutoModelForCausalLM.from_pretrained(
            BASE_ID, trust_remote_code=True, torch_dtype=torch.float32)
    cfg = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA,
        lora_dropout=lora_cfg["lora_dropout"],
        target_modules=list(lora_cfg["target_modules"]),
        bias=lora_cfg.get("bias", "none"),
        task_type=lora_cfg.get("task_type"))
    m = get_peft_model(m, cfg)

    core = _unwrap(m)
    if UNFREEZE_IMG_PROJ:
        core.image_projection.requires_grad_(True)
        for p in core.image_proj_norm.parameters():
            p.requires_grad_(True)

    cache = {"query_embeds": None, "query_mask": None, "reg_loss": None}
    head_holder = [None]

    # Hook pada embedding layer untuk menangkap representasi teks pertanyaan (Query Q)
    emb_layer = core.get_input_embeddings()
    def _emb_hook(module, args, output):
        # Simpan hanya jika panjang sekuens > 2 (yaitu prompt pertanyaan, bukan single-token decoding step)
        if torch.is_tensor(output) and output.dim() == 3 and output.shape[1] > 2:
            cache["query_embeds"] = output.detach()
    emb_hook_handle = emb_layer.register_forward_hook(_emb_hook)

    orig_encode = core._encode_image
    def wrapped_encode_image(pixel_values):
        out = orig_encode(pixel_values)
        h_mod = head_holder[0]
        if h_mod is not None:
            bridged, reg_loss = h_mod.forward_bridge(
                out,
                query_embeds=cache.get("query_embeds"),
                query_mask=cache.get("query_mask")
            )
            cache["feats"] = out
            cache["reg_loss"] = reg_loss
            return bridged.to(out.dtype)
        cache["feats"] = out
        cache["reg_loss"] = torch.tensor(0.0, device=out.device)
        return out

    core._encode_image = wrapped_encode_image
    m._feat_cache = cache
    object.__setattr__(m, "_core_ref", [core])
    object.__setattr__(m, "_head_ref", head_holder)
    object.__setattr__(m, "_emb_hook_ref", [emb_hook_handle])
    return m.to(dev)

model = build_model()
core = model._core_ref[0]
D_VIS = core.image_projection.shape[1]
head = make_head(D_VIS)
model._head_ref[0] = head

model.print_trainable_parameters()
print("GeoFiLM module params :", sum(p.numel() for p in head.parameters()))
print("Active variant        :", VARIANT_KIND)

REG_WEIGHT = 0.02
LR_GROUPS  = {"lora": HP["lr"], "head": HP["lr"] * 50, "head_lr_mult": 50}
print("REG_WEIGHT            :", REG_WEIGHT, "| LR_GROUPS:", LR_GROUPS)

modeling_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base-ft:
- modeling_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/463M [00:00<?, ?B/s]

trainable params: 5,191,776 || all params: 235,817,824 || trainable%: 2.2016
GeoFiLM module params : 416418
Active variant        : M3_StaticGate
REG_WEIGHT            : 0.02 | LR_GROUPS: {'lora': 1e-05, 'head': 0.0005, 'head_lr_mult': 50}


# SMOKE TEST — Uji Cepat Pra-Training (~5 Detik)
<a id="smoke-test"></a>
> [⬆️ Kembali ke Navigasi Cepat](#top) | [Lanjut ke PART B (Mulai Run Baru)](#part-b)

## [A11] Smoke Test Cepat Pra-Training

In [10]:
# [A11] Smoke Test Cepat Pra-Training (Verifikasi Forward, Query-FiLM Hook, Gradien DaViT, & fp16)
print("=" * 72)
print(f"  MEMULAI SMOKE TEST PRA-TRAINING UNTUK VARIAN: {VARIANT_KIND}")
print("=" * 72)

model.train(); head.train()
model.zero_grad(set_to_none=True); head.zero_grad(set_to_none=True)
model._feat_cache.clear()

batch_smk = collate([ds_train[i] for i in range(4)])
batch_smk = {k: (v.to(dev, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch_smk.items()}
model._feat_cache["query_mask"] = batch_smk["attention_mask"]

use_fp16 = HP.get("fp16", True) and (dev == "cuda")
smk_scaler = torch.amp.GradScaler("cuda", enabled=use_fp16)

with torch.amp.autocast("cuda", enabled=use_fp16):
    out_smk = model(
        input_ids=batch_smk["input_ids"],
        pixel_values=batch_smk["pixel_values"],
        attention_mask=batch_smk["attention_mask"],
        labels=batch_smk["labels"]
    )
    reg_smk = model._feat_cache["reg_loss"]
    if reg_smk is None:
        reg_smk = torch.tensor(0.0, device=dev)
    tot_smk = out_smk.loss + REG_WEIGHT * reg_smk

smk_scaler.scale(tot_smk).backward()
smk_scaler.unscale_(torch.optim.AdamW(list(model.parameters()) + list(head.parameters()), lr=1e-5))

# Gradient clipping test
vis_params = [p for n, p in model.named_parameters() if p.grad is not None and any(k in n for k in ["vision_tower", "qkv", "image", "conv"])]
head_params = [p for p in head.parameters() if p.grad is not None]

norm_vis = torch.nn.utils.clip_grad_norm_(vis_params, max_norm=1.0)
norm_head = torch.nn.utils.clip_grad_norm_(head_params, max_norm=1.0)

assert torch.isfinite(tot_smk), "Loss bernilai NaN/Inf!"
assert model._feat_cache.get("query_embeds") is not None, "Query embedding hook tidak terpanggil!"
assert norm_vis > 1e-7 and norm_head > 1e-7, f"Gradien tidak mengalir (vis={norm_vis}, head={norm_head})!"

print(f"  ✓ [1/3] Forward & Query Hook OK | L_VQA: {out_smk.loss.item():.4f} | L_struct: {reg_smk.item():.4f}")
print(f"  ✓ [2/3] Gradient Flow OK        | DaViT Grad Norm: {float(norm_vis):.4f} | GeoFiLM Grad Norm: {float(norm_head):.4f}")
print(f"  ✓ [3/3] Mixed Precision fp16 OK | Total Loss: {tot_smk.item():.4f} | Clip Max-Norm: 1.0 (Stabil)")

model.zero_grad(set_to_none=True); head.zero_grad(set_to_none=True)
model._feat_cache.clear()
del batch_smk, out_smk, reg_smk, tot_smk, smk_scaler
import gc; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("=" * 72)
print("  ✓ SEMUA SMOKE TEST LULUS DENGAN GRADIENT CLIPPING! Model 100% Siap untuk Training.")
print("=" * 72)

  MEMULAI SMOKE TEST PRA-TRAINING UNTUK VARIAN: M3_StaticGate
  ✓ [1/3] Forward & Query Hook OK | L_VQA: 5.5274 | L_struct: 0.0215
  ✓ [2/3] Gradient Flow OK        | DaViT Grad Norm: inf | GeoFiLM Grad Norm: 0.2538
  ✓ [3/3] Mixed Precision fp16 OK | Total Loss: 5.5278 | Clip Max-Norm: 1.0 (Stabil)
  ✓ SEMUA SMOKE TEST LULUS DENGAN GRADIENT CLIPPING! Model 100% Siap untuk Training.


# PART B — FRESH RUN SETUP (Jalankan untuk Memulai Run Baru dari Step 0)
<a id="part-b"></a>
> [⬆️ Kembali ke Navigasi Cepat](#top) | [Lompat ke PART D (Training)](#part-d)

## [B1] Reseed + rebuild model

In [11]:
# [B1] Reseed + rebuild model bersih untuk run FRESH
import gc
from datetime import datetime

if 'model' in globals():
    model.zero_grad(set_to_none=True)
    if hasattr(model, '_feat_cache'):
        model._feat_cache.clear()
if 'head' in globals():
    head.zero_grad(set_to_none=True)
del model, head
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

model = build_model()
core = model._core_ref[0]
D_VIS = core.image_projection.shape[1]
head = make_head(D_VIS)
model._head_ref[0] = head

total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 16.0
EFFECTIVE_BS = HP["batch_size"] * HP["grad_accum"]
MICRO_BS     = 16 if (torch.cuda.is_available() and total_vram >= 30.0) else 4
GRAD_ACCUM   = EFFECTIVE_BS // MICRO_BS
assert MICRO_BS * GRAD_ACCUM == EFFECTIVE_BS

RUN_TAG  = datetime.now().strftime("%Y%m%d-%H%M%S")
RUN_NAME = f"{RUN_TAG}_{VARIANT_KIND}_r{LORA_R}_s{SEED}"
RUN_DIR  = OUTDIR / RUN_NAME
(RUN_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(RUN_DIR / "history").mkdir(parents=True, exist_ok=True)

CONFIG = {
    "run_name": RUN_NAME,
    "started_at": datetime.now().isoformat(timespec="seconds"),
    "variant_kind": VARIANT_KIND,
    "ablation_flags": ABLATION_CONFIGS[VARIANT_KIND],
    "seed": SEED,
    "reg_weight": REG_WEIGHT,
    "lr_groups": LR_GROUPS,
    "hp": HP,
    "effective_batch": EFFECTIVE_BS,
    "micro_batch": MICRO_BS,
    "grad_accum": GRAD_ACCUM,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "n_head_params": int(sum(p.numel() for p in head.parameters())),
    "unfreeze_image_projection": UNFREEZE_IMG_PROJ,
}
(RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2))
print("✓ Fresh model built & RUN_DIR created ->", RUN_DIR)
print(f"  Effective batch = {EFFECTIVE_BS} (micro {MICRO_BS} x accum {GRAD_ACCUM})")

✓ Fresh model built & RUN_DIR created -> /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Paper2_GeoFiLM_r32/20261031-235417_M3_StaticGate_r32_s42
  Effective batch = 32 (micro 16 x accum 2)


# PART C — RESUME ONLY (Jalankan HANYA Jika Training Terputus di Tengah Jalan)
<a id="part-c"></a>
> [⬆️ Kembali ke Navigasi Cepat](#top) | [Lompat ke PART D (Training)](#part-d)

In [ ]:
# [C1] Cari & arahkan ke folder run yang terputus sesuai VARIANT_KIND
cands = []
for p in sorted(OUTDIR.glob("*")):
    if p.is_dir() and (p / "resume/state.pt").exists() and (p / "resume/adapter_model.safetensors").exists():
        d = torch.load(p / "resume/state.pt", map_location="cpu", weights_only=False)
        cands.append(p)
        print(p.name, "| step", d["global_step"], "| epoch", d["epoch"])
assert cands, f"Tidak ada run dengan resume slot di {OUTDIR}"

match = [p for p in cands if VARIANT_KIND in p.name]
RUN_DIR = match[-1] if match else cands[-1]
CONFIG = json.loads((RUN_DIR / "config.json").read_text())
MICRO_BS, GRAD_ACCUM = CONFIG["micro_batch"], CONFIG["grad_accum"]
print("\nRESUME TARGET ->", RUN_DIR.name)

# PART D — TRAINING LOOP UTAMA (5 Epoch)
<a id="part-d"></a>
> [⬆️ Kembali ke Navigasi Cepat](#top) | [Lompat ke PART E (Evaluasi Test Set)](#part-e)

## [D1] Setup DataLoaders, Optimizer, Scheduler, Checkpointing, & Fungsi Validasi VQA Loss

In [12]:
# [D1] Setup DataLoaders, Optimizer, Scheduler, Checkpointing, & Fungsi Validasi VQA Loss
from transformers import get_scheduler
from tqdm.auto import tqdm
import collections

TRAIN_LOG  = RUN_DIR / "history/train.jsonl"
VAL_LOG    = RUN_DIR / "history/val.jsonl"
CKPT_DIR   = RUN_DIR / "checkpoints"
RESUME_DIR = RUN_DIR / "resume"
RESUME_DIR.mkdir(parents=True, exist_ok=True)

def log_line(path, d):
    with open(path, "a") as f:
        f.write(json.dumps(d) + "\n")

def save_ckpt(step, tag=None):
    td = CKPT_DIR / (tag if tag else f"step-{step:06d}")
    td.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(td))
    torch.save(head.state_dict(), td / "geofilm_head.pt")
    core = model._core_ref[0]
    torch.save({
        "image_projection": core.image_projection.data.clone(),
        "image_proj_norm":  core.image_proj_norm.state_dict()
    }, td / "image_proj.pt")
    print(f"saved checkpoint -> {td.name}")

def save_resume(step, epoch):
    tmp = RESUME_DIR / "state.pt.tmp"
    torch.save({
        "global_step": step,
        "epoch": epoch,
        "optimizer": opt.state_dict(),
        "scheduler": sched.state_dict(),
        "scaler": scaler.state_dict(),
        "rng": {
            "python": random.getstate(),
            "numpy":  np.random.get_state(),
            "torch":  torch.get_rng_state(),
            "cuda":   torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        },
        "head_state": head.state_dict(),
        "saved_at": datetime.now().isoformat(timespec="seconds"),
    }, tmp)
    os.replace(tmp, RESUME_DIR / "state.pt")
    model.save_pretrained(str(RESUME_DIR))
    core = model._core_ref[0]
    torch.save({
        "image_projection": core.image_projection.data.clone(),
        "image_proj_norm":  core.image_proj_norm.state_dict()
    }, RESUME_DIR / "image_proj.pt")

dl_tr = DataLoader(ds_train, batch_size=MICRO_BS, shuffle=True,
                   collate_fn=collate, num_workers=4 if IS_COLAB else 0, pin_memory=True)
dl_va = DataLoader(ds_val, batch_size=MICRO_BS, shuffle=False,
                   collate_fn=collate, num_workers=2 if IS_COLAB else 0)

opt = torch.optim.AdamW([
    {"params": [p for p in model.parameters() if p.requires_grad],
     "lr": LR_GROUPS["lora"], "weight_decay": HP["weight_decay"]},
    {"params": list(head.parameters()),
     "lr": LR_GROUPS["head"], "weight_decay": HP["weight_decay"]}
])

STEPS_PER_EPOCH = len(dl_tr) // GRAD_ACCUM
TOTAL_STEPS     = int(STEPS_PER_EPOCH * HP["epochs"])
WARMUP_STEPS    = int(TOTAL_STEPS * HP["warmup_ratio"])
print(f"steps/epoch: {STEPS_PER_EPOCH} | total: {TOTAL_STEPS} | warmup: {WARMUP_STEPS}")

sched = get_scheduler(
    HP["scheduler"], optimizer=opt,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_STEPS)

scaler = torch.amp.GradScaler("cuda", enabled=HP["fp16"] and (dev == "cuda"))

def validate_vqa(step, epoch, max_batches=60):
    model.eval(); head.eval()
    v_losses, r_losses = [], []
    with torch.no_grad():
        for k, bb in enumerate(dl_va):
            if max_batches and k >= max_batches:
                break
            bb = {k: (v.to(dev, non_blocking=True) if torch.is_tensor(v) else v) for k, v in bb.items()}
            model._feat_cache["query_mask"] = bb.get("attention_mask")
            with torch.amp.autocast("cuda", enabled=HP["fp16"] and (dev == "cuda")):
                o = model(input_ids=bb["input_ids"], pixel_values=bb["pixel_values"],
                          attention_mask=bb.get("attention_mask"), labels=bb["labels"])
                rl = model._feat_cache.get("reg_loss")
                v_losses.append(float(o.loss.item()))
                if rl is not None:
                    r_losses.append(float(rl.item()))
            model._feat_cache.clear()
    model.train(); head.train()
    rec = {
        "global_step": step, "epoch": epoch,
        "val_vqa_loss": round(float(np.mean(v_losses)), 4) if v_losses else None,
        "val_reg_loss": round(float(np.mean(r_losses)), 4) if r_losses else None,
        "pe_scale": round(float(torch.tanh(head.pe_scale).item()), 4),
        "at": datetime.now().isoformat(timespec="seconds")
    }
    log_line(VAL_LOG, rec)
    print(f"\n[VAL step {step} ep {epoch}] val_L_VQA={rec['val_vqa_loss']:.4f} | val_L_struct={rec['val_reg_loss']:.4f} | pe_scale={rec['pe_scale']:.4f}")
    return rec

steps/epoch: 4032 | total: 20160 | warmup: 2016


## [D2] Jalankan Loop Training 5 Epoch

In [13]:
# [D2] Jalankan Loop Training 5 Epoch (dengan Auto-Restore saat Resume)
LOG_EVERY  = 50
VAL_EVERY  = 500
SAVE_EVERY = 500

gstep = 0
start_epoch = 0

resume_state = RUN_DIR / "resume/state.pt"
if resume_state.exists():
    d = torch.load(resume_state, map_location="cpu", weights_only=False)
    gstep = d["global_step"]
    start_epoch = d["epoch"]
    opt.load_state_dict(d["optimizer"])
    sched.load_state_dict(d["scheduler"])
    scaler.load_state_dict(d["scaler"])
    random.setstate(d["rng"]["python"])
    np.random.set_state(d["rng"]["numpy"])
    torch.set_rng_state(d["rng"]["torch"])
    if torch.cuda.is_available() and d["rng"]["cuda"] is not None:
        torch.cuda.set_rng_state_all(d["rng"]["cuda"])
    head.load_state_dict(d["head_state"])
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file
    set_peft_model_state_dict(model, load_file(str(RUN_DIR / "resume/adapter_model.safetensors")))
    if (RUN_DIR / "resume/image_proj.pt").exists():
        ip_st = torch.load(RUN_DIR / "resume/image_proj.pt", map_location=dev)
        core = model._core_ref[0]
        core.image_projection.data.copy_(ip_st["image_projection"])
        core.image_proj_norm.load_state_dict(ip_st["image_proj_norm"])
    print(f"✓ Resumed dari step {gstep}, epoch {start_epoch}")

t0 = time.time()
validate_vqa(gstep, start_epoch, max_batches=40)

for ep in range(start_epoch, int(math.ceil(HP["epochs"]))):
    model.train(); head.train()
    pbar = tqdm(dl_tr, desc=f"{VARIANT_KIND} epoch {ep}")
    for i, bb in enumerate(pbar):
        bb = {k: (v.to(dev, non_blocking=True) if torch.is_tensor(v) else v) for k, v in bb.items()}
        model._feat_cache["query_mask"] = bb.get("attention_mask")
        with torch.amp.autocast("cuda", enabled=HP["fp16"] and (dev == "cuda")):
            o = model(input_ids=bb["input_ids"], pixel_values=bb["pixel_values"],
                      attention_mask=bb.get("attention_mask"), labels=bb["labels"])
            rl = model._feat_cache.get("reg_loss")
            if rl is None:
                rl = torch.tensor(0.0, device=dev)
            loss = (o.loss + REG_WEIGHT * rl) / GRAD_ACCUM

        scaler.scale(loss).backward()
        model._feat_cache.clear()

        if (i + 1) % GRAD_ACCUM == 0:
            # Proteksi stabilitas numerik: unscale dan clip grad norm ke 1.0 sebelum optimizer step
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            torch.nn.utils.clip_grad_norm_(head.parameters(), max_norm=1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True); sched.step(); gstep += 1

            if gstep % LOG_EVERY == 0:
                lv = float(o.loss.item())
                lr_val = float(rl.item())
                log_line(TRAIN_LOG, {
                    "global_step": gstep, "epoch": ep,
                    "L_vqa": lv, "L_struct": lr_val,
                    "pe_scale": round(float(torch.tanh(head.pe_scale).item()), 4),
                    "gpu_gb": round(torch.cuda.memory_allocated() / 1e9, 2) if torch.cuda.is_available() else 0.0,
                    "elapsed_s": round(time.time() - t0, 1)
                })
                pbar.set_postfix(vqa=f"{lv:.3f}", reg=f"{lr_val:.4f}")

            if gstep % VAL_EVERY == 0:
                validate_vqa(gstep, ep, max_batches=50)
            if gstep % SAVE_EVERY == 0:
                save_ckpt(gstep); save_resume(gstep, ep)

    validate_vqa(gstep, ep, max_batches=100)
    save_ckpt(gstep, tag=f"epoch-{ep:02d}")
    save_resume(gstep, ep + 1)

save_ckpt(gstep, tag="final")
CONFIG["finished_at"] = datetime.now().isoformat(timespec="seconds")
CONFIG["total_steps"] = gstep
CONFIG["wall_clock_hours"] = round((time.time() - t0) / 3600, 2)
CONFIG["status"] = "ok"
(RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2))
print(f"\n✓ Training {VARIANT_KIND} selesai dalam {CONFIG['wall_clock_hours']} jam -> {RUN_DIR}")


[VAL step 0 ep 0] val_L_VQA=5.0737 | val_L_struct=0.0215 | pe_scale=0.0000


M3_StaticGate epoch 0:   0%|          | 0/8064 [00:00<?, ?it/s]


[VAL step 500 ep 0] val_L_VQA=3.1395 | val_L_struct=0.0014 | pe_scale=0.0336
saved checkpoint -> step-000500

[VAL step 1000 ep 0] val_L_VQA=2.3954 | val_L_struct=0.0012 | pe_scale=0.0599
saved checkpoint -> step-001000

[VAL step 1500 ep 0] val_L_VQA=1.9800 | val_L_struct=0.0012 | pe_scale=0.1200
saved checkpoint -> step-001500

[VAL step 2000 ep 0] val_L_VQA=1.6529 | val_L_struct=0.0013 | pe_scale=0.2376
saved checkpoint -> step-002000

[VAL step 2500 ep 0] val_L_VQA=1.4189 | val_L_struct=0.0012 | pe_scale=0.3375
saved checkpoint -> step-002500

[VAL step 3000 ep 0] val_L_VQA=1.2725 | val_L_struct=0.0010 | pe_scale=0.4070
saved checkpoint -> step-003000

[VAL step 3500 ep 0] val_L_VQA=1.1842 | val_L_struct=0.0010 | pe_scale=0.4544
saved checkpoint -> step-003500

[VAL step 4000 ep 0] val_L_VQA=1.1170 | val_L_struct=0.0011 | pe_scale=0.4871
saved checkpoint -> step-004000

[VAL step 4032 ep 0] val_L_VQA=1.1372 | val_L_struct=0.0006 | pe_scale=0.4897
saved checkpoint -> epoch-00


M3_StaticGate epoch 1:   0%|          | 0/8064 [00:00<?, ?it/s]


[VAL step 4500 ep 1] val_L_VQA=1.0630 | val_L_struct=0.0009 | pe_scale=0.5134
saved checkpoint -> step-004500

[VAL step 5000 ep 1] val_L_VQA=1.0284 | val_L_struct=0.0007 | pe_scale=0.5333
saved checkpoint -> step-005000

[VAL step 5500 ep 1] val_L_VQA=1.0036 | val_L_struct=0.0007 | pe_scale=0.5477
saved checkpoint -> step-005500

[VAL step 6000 ep 1] val_L_VQA=0.9745 | val_L_struct=0.0008 | pe_scale=0.5611
saved checkpoint -> step-006000

[VAL step 6500 ep 1] val_L_VQA=0.9485 | val_L_struct=0.0009 | pe_scale=0.5713
saved checkpoint -> step-006500

[VAL step 7000 ep 1] val_L_VQA=0.9285 | val_L_struct=0.0008 | pe_scale=0.5802
saved checkpoint -> step-007000

[VAL step 7500 ep 1] val_L_VQA=0.9145 | val_L_struct=0.0006 | pe_scale=0.5882
saved checkpoint -> step-007500

[VAL step 8000 ep 1] val_L_VQA=0.8963 | val_L_struct=0.0007 | pe_scale=0.5959
saved checkpoint -> step-008000

[VAL step 8064 ep 1] val_L_VQA=0.9146 | val_L_struct=0.0008 | pe_scale=0.5961
saved checkpoint -> epoch-01


M3_StaticGate epoch 2:   0%|          | 0/8064 [00:00<?, ?it/s]


[VAL step 8500 ep 2] val_L_VQA=0.8874 | val_L_struct=0.0006 | pe_scale=0.6021
saved checkpoint -> step-008500

[VAL step 9000 ep 2] val_L_VQA=0.8693 | val_L_struct=0.0005 | pe_scale=0.6068
saved checkpoint -> step-009000

[VAL step 9500 ep 2] val_L_VQA=0.8598 | val_L_struct=0.0006 | pe_scale=0.6116
saved checkpoint -> step-009500

[VAL step 10000 ep 2] val_L_VQA=0.8505 | val_L_struct=0.0005 | pe_scale=0.6144
saved checkpoint -> step-010000

[VAL step 10500 ep 2] val_L_VQA=0.8400 | val_L_struct=0.0005 | pe_scale=0.6183
saved checkpoint -> step-010500

[VAL step 11000 ep 2] val_L_VQA=0.8334 | val_L_struct=0.0008 | pe_scale=0.6205
saved checkpoint -> step-011000

[VAL step 11500 ep 2] val_L_VQA=0.8282 | val_L_struct=0.0006 | pe_scale=0.6244
saved checkpoint -> step-011500

[VAL step 12000 ep 2] val_L_VQA=0.8175 | val_L_struct=0.0005 | pe_scale=0.6241
saved checkpoint -> step-012000

[VAL step 12096 ep 2] val_L_VQA=0.8269 | val_L_struct=0.0006 | pe_scale=0.6242
saved checkpoint -> epoch-0

M3_StaticGate epoch 3:   0%|          | 0/8064 [00:00<?, ?it/s]


[VAL step 12500 ep 3] val_L_VQA=0.8150 | val_L_struct=0.0005 | pe_scale=0.6255
saved checkpoint -> step-012500

[VAL step 13000 ep 3] val_L_VQA=0.8097 | val_L_struct=0.0007 | pe_scale=0.6271
saved checkpoint -> step-013000

[VAL step 13500 ep 3] val_L_VQA=0.8069 | val_L_struct=0.0007 | pe_scale=0.6293
saved checkpoint -> step-013500

[VAL step 14000 ep 3] val_L_VQA=0.7978 | val_L_struct=0.0006 | pe_scale=0.6308
saved checkpoint -> step-014000

[VAL step 14500 ep 3] val_L_VQA=0.7949 | val_L_struct=0.0005 | pe_scale=0.6305
saved checkpoint -> step-014500

[VAL step 15000 ep 3] val_L_VQA=0.7955 | val_L_struct=0.0006 | pe_scale=0.6315
saved checkpoint -> step-015000

[VAL step 15500 ep 3] val_L_VQA=0.7868 | val_L_struct=0.0005 | pe_scale=0.6307
saved checkpoint -> step-015500

[VAL step 16000 ep 3] val_L_VQA=0.7807 | val_L_struct=0.0005 | pe_scale=0.6308
saved checkpoint -> step-016000

[VAL step 16128 ep 3] val_L_VQA=0.7941 | val_L_struct=0.0005 | pe_scale=0.6314
saved checkpoint -> epoc

M3_StaticGate epoch 4:   0%|          | 0/8064 [00:00<?, ?it/s]


[VAL step 16500 ep 4] val_L_VQA=0.7803 | val_L_struct=0.0005 | pe_scale=0.6317
saved checkpoint -> step-016500

[VAL step 17000 ep 4] val_L_VQA=0.7776 | val_L_struct=0.0005 | pe_scale=0.6312
saved checkpoint -> step-017000

[VAL step 17500 ep 4] val_L_VQA=0.7771 | val_L_struct=0.0007 | pe_scale=0.6319
saved checkpoint -> step-017500

[VAL step 18000 ep 4] val_L_VQA=0.7740 | val_L_struct=0.0006 | pe_scale=0.6320
saved checkpoint -> step-018000

[VAL step 18500 ep 4] val_L_VQA=0.7742 | val_L_struct=0.0006 | pe_scale=0.6320
saved checkpoint -> step-018500

[VAL step 19000 ep 4] val_L_VQA=0.7727 | val_L_struct=0.0005 | pe_scale=0.6324
saved checkpoint -> step-019000

[VAL step 19500 ep 4] val_L_VQA=0.7716 | val_L_struct=0.0005 | pe_scale=0.6325
saved checkpoint -> step-019500

[VAL step 20000 ep 4] val_L_VQA=0.7705 | val_L_struct=0.0005 | pe_scale=0.6323
saved checkpoint -> step-020000

[VAL step 20160 ep 4] val_L_VQA=0.7795 | val_L_struct=0.0005 | pe_scale=0.6323
saved checkpoint -> epoc

# PART E — INFERENSI GENERATIF 15.955 BARIS TEST SET & TABEL ABLASI PAPER 2
<a id="part-e"></a>
> [⬆️ Kembali ke Navigasi Cepat](#top)
>
> Bagian ini menjalankan inferensi generatif (`num_beams=3`, `max_new_tokens=48`) pada seluruh **15.955 QA test set**, menyimpan `eval/predictions.json` (dengan fitur *auto-resume* per 200 sampel), lalu mengevaluasi metrik deterministik Level 1 per kategori, metrik lokasi Jaccard/Grid, dan metrik NLG (ROUGE, BLEU, METEOR, CHRF++). File `predictions.json` yang dihasilkan juga langsung kompatibel 100% dengan skrip `07_dual_llm_judge_evaluation.ipynb` untuk penilaian Qwen2.5-7B & Llama-3.1-8B Judge!

## [E1] Muat Test Set

In [14]:
# [E1] Muat Test Set (15.955 QA pairs) & Pilih Folder Run yang Akan Dievaluasi
from pathlib import Path
import os, json
from datasets import load_from_disk
from peft import PeftModel

if "BASE" not in globals():
    BASE = Path("/content/drive/MyDrive/DOKTORAL/KvasirVQAX1") if Path("/content/drive/MyDrive/DOKTORAL/KvasirVQAX1").exists() else Path(r"g:/My Drive/DOKTORAL/KvasirVQAX1")
if "OUTDIR" not in globals():
    OUTDIR = BASE / "experiments/Paper2_GeoFiLM_r32"
if "VARIANT_KIND" not in globals():
    VARIANT_KIND = "M4_GeoFiLM"

te = load_from_disk(str(BASE / "test")).to_pandas()
print(f"✓ Total test rows loaded: {len(te)}")

# Pilih folder run terbaru untuk VARIANT_KIND aktif
eval_cands = [p for p in sorted(OUTDIR.glob("*")) if p.is_dir() and (p / "checkpoints/final").exists()]
match_eval = [p for p in eval_cands if VARIANT_KIND in p.name]
TARGET_RUN = match_eval[-1] if match_eval else (eval_cands[-1] if eval_cands else globals().get("RUN_DIR", None))
print(f"✓ TARGET_RUN untuk evaluasi: {TARGET_RUN}")


✓ Total test rows loaded: 15955
✓ TARGET_RUN untuk evaluasi: /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Paper2_GeoFiLM_r32/20261031-235417_M3_StaticGate_r32_s42


## [E2] Load Checkpoint Final (LoRA + GeoFiLMHead + Unfrozen image_projection) & Jalankan Inferensi

In [15]:
# [E2] Load Checkpoint Final (LoRA + GeoFiLMHead + Unfrozen image_projection) & Jalankan Inferensi
from unittest.mock import patch
from pathlib import Path
import json, re, os, torch
from transformers import AutoProcessor, AutoModelForCausalLM
from peft import PeftModel
from PIL import Image
from tqdm.auto import tqdm

# 1. Pastikan variabel lingkungan & dependensi dasar tersedia secara mandiri
if "BASE" not in globals():
    BASE = Path("/content/drive/MyDrive/DOKTORAL/KvasirVQAX1") if Path("/content/drive/MyDrive/DOKTORAL/KvasirVQAX1").exists() else Path(r"g:/My Drive/DOKTORAL/KvasirVQAX1")
if "IMG_DIR" not in globals() or not IMG_DIR.exists():
    for cand_dir in [Path("/content/all_images"), BASE / "all_images"]:
        if cand_dir.exists():
            IMG_DIR = cand_dir
            break
if "ABLATION_CONFIGS" not in globals():
    ABLATION_CONFIGS = {
        "M1_GeoPE":       dict(use_geope=True,  use_diff=False, gate_mode="additive"),
        "M2_GeoDiff":     dict(use_geope=True,  use_diff=True,  gate_mode="additive"),
        "M3_StaticGate":  dict(use_geope=True,  use_diff=True,  gate_mode="scalar"),
        "M4_GeoFiLM":     dict(use_geope=True,  use_diff=True,  gate_mode="film"),
    }
if "GeoFiLMModule" not in globals():
    raise RuntimeError("Class GeoFiLMModule belum dimuat ke memori! Silakan jalankan Cell [A8] terlebih dahulu.")

dev = "cuda" if torch.cuda.is_available() else "cpu"
BASE_ID = "microsoft/Florence-2-base-ft"
PROMPT_PREFIX = "<VQA> "
GEN_CFG = {"max_new_tokens": 64}
strip_jpg = lambda s: re.sub(r"\.(jpg|jpeg|png)$", "", str(s), flags=re.I)

def _skip_flash(name, *args, **kwargs):
    if "flash_attn" in str(name): return None
    return orig_get_imports(name, *args, **kwargs)

import transformers.dynamic_module_utils as dmu
orig_get_imports = dmu.get_imports

def _unwrap_eval(m):
    seen = set()
    def walk(x, depth=0):
        if depth > 6 or id(x) in seen: return None
        seen.add(id(x))
        if "_encode_image" in type(x).__dict__: return x
        for name in ("base_model", "model"):
            child = getattr(x, name, None)
            if child is not None and child is not x:
                r = walk(child, depth + 1)
                if r is not None: return r
        return None
    found = walk(m)
    assert found is not None, "could not locate the module defining _encode_image"
    return found

def load_geofilm_for_eval(run_dir):
    cfg = json.loads((run_dir / "config.json").read_text())
    v_kind = cfg.get("variant_kind", "M4_GeoFiLM")
    ckpt = run_dir / "checkpoints/final"
    print(f"Loading checkpoint from: {ckpt} (Variant: {v_kind})...")

    with patch("transformers.dynamic_module_utils.get_imports", _skip_flash):
        proc = AutoProcessor.from_pretrained(BASE_ID, trust_remote_code=True)
        base = AutoModelForCausalLM.from_pretrained(BASE_ID, trust_remote_code=True, torch_dtype=torch.float16)
    m = PeftModel.from_pretrained(base, str(ckpt)).to(dev).eval()
    core = _unwrap_eval(m)

    ip_path = ckpt / "image_proj.pt"
    if ip_path.exists():
        ip_st = torch.load(ip_path, map_location=dev)
        core.image_projection.data.copy_(ip_st["image_projection"].to(device=dev, dtype=core.image_projection.dtype))
        core.image_proj_norm.load_state_dict(ip_st["image_proj_norm"])
        print("  ✓ Restored unfrozen image_projection & image_proj_norm.")

    d_vis = core.image_projection.shape[1]
    ablation_kwargs = cfg.get("ablation_flags", ABLATION_CONFIGS.get(v_kind, dict(use_geope=True, use_diff=True, gate_mode="film")))
    h_mod = GeoFiLMModule(d_in=d_vis, **ablation_kwargs).to(device=dev, dtype=torch.float16)
    h_mod.load_state_dict(torch.load(ckpt / "geofilm_head.pt", map_location=dev))
    h_mod.to(device=dev, dtype=torch.float16).eval()
    print("  ✓ Restored GeoFiLMModule weights.")

    cache = {"query_embeds": None, "query_mask": None}
    emb_layer = core.get_input_embeddings()
    def _emb_hook(module, args, output):
        if torch.is_tensor(output) and output.dim() == 3 and output.shape[1] > 2:
            cache["query_embeds"] = output.detach()
    emb_layer.register_forward_hook(_emb_hook)

    orig_encode = core._encode_image
    def wrapped_encode(pixel_values):
        out = orig_encode(pixel_values)
        bridged, _ = h_mod.forward_bridge(out, query_embeds=cache.get("query_embeds"), query_mask=cache.get("query_mask"))
        return bridged.to(out.dtype)
    core._encode_image = wrapped_encode
    m._feat_cache = cache
    return m, proc, h_mod, core

eval_model, eval_proc, eval_head, eval_core = load_geofilm_for_eval(TARGET_RUN)

EVAL_DIR = TARGET_RUN / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
PRED_PATH = EVAL_DIR / "predictions.json"

predictions = {}
if PRED_PATH.exists():
    try:
        predictions = {int(k): v for k, v in json.loads(PRED_PATH.read_text()).items()}
        print(f"Melanjutkan dari cache: {len(predictions)} / {len(te)} baris sudah selesai.")
    except Exception:
        predictions = {}

indices_to_run = [i for i in range(len(te)) if i not in predictions]
print(f"Sisa baris yang akan di-inferensi: {len(indices_to_run)}")

def resolve_image(img_id):
    stem = strip_jpg(img_id)
    cand = IMG_DIR / f"{stem}.jpg"
    if cand.exists(): return cand
    matches = list(IMG_DIR.glob(f"{stem}.*"))
    return matches[0] if matches else None

@torch.no_grad()
def run_generate_batch(prompts_list, imgs_list):
    inputs = eval_proc(text=prompts_list, images=imgs_list, return_tensors="pt", padding=True)
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

    # Isi cache query_embeds & query_mask sebelum _encode_image dipanggil oleh .generate()
    q_emb = eval_core.get_input_embeddings()(inputs["input_ids"])
    eval_model._feat_cache["query_embeds"] = q_emb.detach()
    eval_model._feat_cache["query_mask"] = inputs.get("attention_mask")

    # JANGAN kirim attention_mask ke .generate() agar Florence-2 menggabungkan mask 577+L secara internal
    gen_ids = eval_model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        **GEN_CFG
    )
    return [ans.strip() for ans in eval_proc.batch_decode(gen_ids, skip_special_tokens=True)]

BATCH_INF = 16 if (torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory / 1e9 >= 30.0) else 8

for start_i in tqdm(range(0, len(indices_to_run), BATCH_INF), desc=f"Evaluating {TARGET_RUN.name[:25]}"):
    batch_indices = indices_to_run[start_i:start_i + BATCH_INF]
    imgs, prompts, valid_idx = [], [], []
    for idx in batch_indices:
        row = te.iloc[idx]
        p_img = resolve_image(row["img_id"])
        if p_img is None:
            predictions[idx] = ""
            continue
        try:
            img = Image.open(p_img).convert("RGB")
        except Exception:
            predictions[idx] = ""
            continue
        imgs.append(img)
        prompts.append(PROMPT_PREFIX + str(row["question"]).strip())
        valid_idx.append(idx)

    if not valid_idx:
        continue

    try:
        decoded = run_generate_batch(prompts, imgs)
        for idx, ans in zip(valid_idx, decoded):
            predictions[idx] = ans
    except RuntimeError as e:
        torch.cuda.empty_cache()
        for idx, p_txt, im in zip(valid_idx, prompts, imgs):
            predictions[idx] = run_generate_batch([p_txt], [im])[0]

    if (start_i // BATCH_INF) % 25 == 0:
        PRED_PATH.write_text(json.dumps({str(k): predictions[k] for k in sorted(predictions.keys())}, indent=2))

PRED_PATH.write_text(json.dumps({str(k): predictions[k] for k in sorted(predictions.keys())}, indent=2))
print(f"\n✓ Inferensi selesai! Tersimpan di {PRED_PATH} ({len(predictions)} baris).")


Loading checkpoint from: /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Paper2_GeoFiLM_r32/20261031-235417_M3_StaticGate_r32_s42/checkpoints/final (Variant: M3_StaticGate)...
  ✓ Restored unfrozen image_projection & image_proj_norm.
  ✓ Restored GeoFiLMModule weights.
Sisa baris yang akan di-inferensi: 15955


Evaluating 20261031-235417_M3_Static:   0%|          | 0/998 [00:00<?, ?it/s]


✓ Inferensi selesai! Tersimpan di /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Paper2_GeoFiLM_r32/20261031-235417_M3_StaticGate_r32_s42/eval/predictions.json (15955 baris).


## [E3] Evaluasi Cepat & Perbandingan Langsung: M4_GeoFiLM vs Baseline A0 (VQA-only r=32)

In [16]:
# [E3] Evaluasi Cepat & Perbandingan Langsung: M4_GeoFiLM vs Baseline A0 (VQA-only r=32)
import json, re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

if "PRED_PATH" in globals() and Path(PRED_PATH).exists() and ("predictions" not in globals() or not predictions):
    predictions = {int(k): v for k, v in json.loads(Path(PRED_PATH).read_text()).items()}

GRID_CELLS = [
    "upper-left", "upper-center", "upper-right",
    "center-left", "center", "center-right",
    "lower-left", "lower-center", "lower-right"
]

def extract_grid_cells(text):
    t = str(text).lower()
    found = set()
    for cell in GRID_CELLS:
        if re.search(rf"\b{re.escape(cell)}\b", t):
            found.add(cell)
    return found

def token_f1(pred, gold):
    p_toks = re.findall(r"\w+", str(pred).lower())
    g_toks = re.findall(r"\w+", str(gold).lower())
    if not p_toks or not g_toks:
        return float(p_toks == g_toks)
    common = sum((Counter(p_toks) & Counter(g_toks)).values())
    if common == 0:
        return 0.0
    prec = common / len(p_toks)
    rec  = common / len(g_toks)
    return 2 * prec * rec / (prec + rec)

col_cat = next((c for c in ["question_class", "categories", "category", "aspects"] if c in te.columns), "question_class")
_DELIMS = re.compile(r"[,;|]")
def to_label_set(v):
    if v is None: return set()
    if isinstance(v, (list, tuple, set, np.ndarray)):
        return {str(x).strip() for x in v if str(x).strip()}
    s = str(v).strip()
    if s.startswith("[") and s.endswith("]"):
        try: return {str(x).strip() for x in json.loads(s.replace("'", '"'))}
        except Exception: s = s[1:-1]
    return {p.strip().strip('"\'') for p in _DELIMS.split(s) if p.strip()}

te["_cats"] = te[col_cat].map(to_label_set)
te["_pred"] = [predictions.get(i, "") for i in range(len(te))]

# Coba muat prediksi Baseline A0 (VQA-only r=32) jika tersedia untuk komparasi langsung
a0_preds = None
a0_cand = BASE / "experiments/Stage2_CORAL_linear/prediction_rank32/20260916-060029_VQAonly-r32_a0.0_s42/eval/predictions.json"
if a0_cand.exists():
    raw_a0 = json.loads(a0_cand.read_text())
    a0_preds = [raw_a0.get(str(i), "") for i in range(len(te))]
    te["_a0"] = a0_preds

print("=" * 88)
print(f"  HASIL EVALUASI SPASIAL 9-GRID: {TARGET_RUN.name}")
print("=" * 88)
for loc_cat in ["abnormality_location", "landmark_location", "instrument_location"]:
    sub = te[te["_cats"].map(lambda s: loc_cat in s)]
    gold_sets = sub["answer"].map(extract_grid_cells)
    pred_sets = sub["_pred"].map(extract_grid_cells)
    valid_mask = gold_sets.map(len) > 0
    if valid_mask.sum() > 0:
        jacc = [len(g & p) / max(len(g | p), 1) for g, p in zip(gold_sets[valid_mask], pred_sets[valid_mask])]
        rec  = [len(g & p) / max(len(g), 1)     for g, p in zip(gold_sets[valid_mask], pred_sets[valid_mask])]
        generic_rate = sub["_pred"].str.lower().str.contains("scattered across multiple regions").mean()
        line = f"  {loc_cat:22s} (n={len(sub):4d}) | Jaccard: {np.mean(jacc)*100:5.2f}% | Recall: {np.mean(rec)*100:5.2f}% | Hedge Collapse: {generic_rate*100:5.2f}%"
        if a0_preds is not None:
            a0_sets = sub["_a0"].map(extract_grid_cells)
            jacc_a0 = [len(g & p) / max(len(g | p), 1) for g, p in zip(gold_sets[valid_mask], a0_sets[valid_mask])]
            hedge_a0 = sub["_a0"].str.lower().str.contains("scattered across multiple regions").mean()
            line += f"  [vs A0 Jaccard: {np.mean(jacc_a0)*100:5.2f}%, Hedge: {hedge_a0*100:5.2f}%]"
        print(line)

print("\n" + "=" * 88)
print("  TOKEN-F1 PER KATEGORI TARGET (LOKASI, WARNA, REMOVAL STATUS, & OVERALL)")
print("=" * 88)
target_cats = [
    "abnormality_location", "landmark_location", "instrument_location",
    "abnormality_color", "landmark_color", "instrument_color",
    "removal_status", "finding_presence", "polyp_size", "polyp_count"
]
for cat in target_cats:
    sub = te[te["_cats"].map(lambda s: cat in s)]
    if len(sub) == 0: continue
    f1_ours = np.mean([token_f1(p, g) for p, g in zip(sub["_pred"], sub["answer"])]) * 100
    if a0_preds is not None:
        f1_a0 = np.mean([token_f1(p, g) for p, g in zip(sub["_a0"], sub["answer"])]) * 100
        diff = f1_ours - f1_a0
        print(f"  {cat:22s} (n={len(sub):4d}) | GeoFiLM F1: {f1_ours:5.2f}% | A0 F1: {f1_a0:5.2f}% | Delta: {diff:+5.2f}%")
    else:
        print(f"  {cat:22s} (n={len(sub):4d}) | GeoFiLM F1: {f1_ours:5.2f}%")

overall_f1 = np.mean([token_f1(p, g) for p, g in zip(te["_pred"], te["answer"])]) * 100
if a0_preds is not None:
    overall_a0 = np.mean([token_f1(p, g) for p, g in zip(te["_a0"], te["answer"])]) * 100
    print("-" * 88)
    print(f"  {'OVERALL (15,955 QA)':22s}          | GeoFiLM F1: {overall_f1:5.2f}% | A0 F1: {overall_a0:5.2f}% | Delta: {overall_f1 - overall_a0:+5.2f}%")
else:
    print("-" * 88)
    print(f"  {'OVERALL (15,955 QA)':22s}          | GeoFiLM F1: {overall_f1:5.2f}%")

print("\nCatatan: Untuk evaluasi Dual-LLM Judge lengkap (18 kategori x 3 level = 31.622 aspek),")
print(f"gunakan file prediksi: {PRED_PATH}")

  HASIL EVALUASI SPASIAL 9-GRID: 20261031-235417_M3_StaticGate_r32_s42
  abnormality_location   (n=1901) | Jaccard:  2.90% | Recall:  4.19% | Hedge Collapse: 27.30%  [vs A0 Jaccard: 12.05%, Hedge: 39.61%]
  landmark_location      (n=1224) | Jaccard:  0.83% | Recall:  1.39% | Hedge Collapse:  3.02%  [vs A0 Jaccard:  4.48%, Hedge:  4.00%]
  instrument_location    (n=1913) | Jaccard: 14.63% | Recall: 17.00% | Hedge Collapse:  2.51%  [vs A0 Jaccard:  6.22%, Hedge:  3.97%]

  TOKEN-F1 PER KATEGORI TARGET (LOKASI, WARNA, REMOVAL STATUS, & OVERALL)
  abnormality_location   (n=1901) | GeoFiLM F1: 61.29% | A0 F1: 59.69% | Delta: +1.60%
  landmark_location      (n=1224) | GeoFiLM F1: 66.56% | A0 F1: 64.07% | Delta: +2.49%
  instrument_location    (n=1913) | GeoFiLM F1: 68.91% | A0 F1: 68.06% | Delta: +0.85%
  abnormality_color      (n=1862) | GeoFiLM F1: 61.02% | A0 F1: 58.98% | Delta: +2.04%
  landmark_color         (n=  55) | GeoFiLM F1: 56.51% | A0 F1: 57.80% | Delta: -1.29%
  finding_presenc